In [1]:
import geopandas as gpd
import pandas as pd
from shapely import wkt



In [4]:
# Load recreational points
rec = pd.read_csv("dataset/rec_facilities.csv")
rec["geometry"] = rec["POINT"].apply(wkt.loads)
rec_gdf = gpd.GeoDataFrame(rec, geometry="geometry", crs="EPSG:4326")

# Load community boundaries (multipolygons)
comm = pd.read_csv("dataset/community_boundaries.csv")

# Convert the multipolygon WKT to geometry
comm["geometry"] = comm["MULTIPOLYGON"].apply(wkt.loads)
comm_gdf = gpd.GeoDataFrame(comm, geometry="geometry", crs="EPSG:4326")


In [6]:
rec_gdf = rec_gdf.to_crs(comm_gdf.crs)

In [25]:
joined = gpd.sjoin(
    rec_gdf,
    comm_gdf[['NAME', 'SECTOR', 'geometry']],  # keep only relevant columns
    how='left',
    predicate='within'
)

In [34]:
result = joined[['POINT', 'NAME', 'SECTOR']]
result = result.rename(columns={"NAME": "COMMUNITY"})

In [35]:
result.to_csv("rec_facilities_with_neighborhoods.csv", index=False)

In [39]:
res = pd.read_csv('rec_facilities_with_neighborhoods.csv')
res

,POINT,COMMUNITY,SECTOR
0,POINT (-114.0940119 51.1818643),CARRINGTON,NORTH
1,POINT (-113.9649419 51.0413756),FOREST LAWN,EAST
2,POINT (-114.1247929 51.0455075),SHAGANAPPI,CENTRE
3,POINT (-114.1378257 51.0975122),BRENTWOOD,NORTHWEST
4,POINT (-114.1026915 51.0074053),GLENMORE PARK,SOUTH
...,...,...,...
107,POINT (-114.08953 50.9484097),CANYON MEADOWS,SOUTH
108,POINT (-114.0085882 50.9962749),OGDEN,SOUTHEAST
109,POINT (-114.102984 51.057977),WEST HILLHURST,CENTRE
110,POINT (-114.0629633 50.9710322),ACADIA,SOUTH


In [40]:
# Merge on a column
merged = pd.merge(rec, res, on="POINT", how="left")  # left join
# other options for how: 'inner', 'right', 'outer'

# Save to CSV
merged.to_csv("recreation_fat.csv", index=False)

In [53]:
recreation_facilities = pd.read_csv('recreation_fat.csv')
recreation_facilities.head(1)

,COMPLEX_CATEGORY,COMPLEX_NAME,FACILITY_TYPE,FACILITY_NAME,TIER,ADDRESS,TYP_OF_ORG,ORG,STEWARD,OPER,...,DRIVERANG_INDR,SKI_NORDICTRCK,MUNICPAL_OP,NOTPROFIT_OP,PROFIT_OP,PRIVATE_OP,POINT,geometry,COMMUNITY,SECTOR
0,NaN,Carrington Skatepark,Skate Park,NaN,NaN,37 CARRINGTON PZ NW,Municipal,Parks,Parks,Parks,...,N,N,Y,N,N,N,POINT (-114.0940119 51.1818643),POINT (-114.0940119 51.1818643),CARRINGTON,NORTH


In [61]:
columns_to_keep = ['COMPLEX_CATEGORY', 'COMPLEX_NAME', 'FACILITY_TYPE', 'FACILITY_NAME',
     'ADDRESS', 'TYP_OF_ORG', 'ORG', 'STEWARD', 'OPER', 'COMMUNITY', 'SECTOR']

for col in recreation_facilities.columns:
    if col not in columns_to_keep:
        recreation_facilities.drop(col, axis=1, inplace=True)

In [64]:
recreation_facilities

,COMPLEX_CATEGORY,COMPLEX_NAME,FACILITY_TYPE,FACILITY_NAME,ADDRESS,TYP_OF_ORG,ORG,STEWARD,OPER,COMMUNITY,SECTOR
0,NaN,Carrington Skatepark,Skate Park,NaN,37 CARRINGTON PZ NW,Municipal,Parks,Parks,Parks,CARRINGTON,NORTH
1,Arenas (except Leisure Centre arenas),Ernie Starr Arena,Arena,Ice,4808 14 AV SE,Municipal,Recreation,Recreation,Recreation,FOREST LAWN,EAST
2,Golf Courses,Shaganappi Point Golf Course,Golf Course - Municipal,18 Hole Course,1200 26 ST SW,Municipal,Parks,Recreation,Parks,SHAGANAPPI,CENTRE
3,NaN,Brentwood Sportsplex,Arena,NaN,1520B NORTHMOUNT DR NW,Not-for-Profit,CA,Recreation,Brentwood CA,BRENTWOOD,NORTHWEST
4,Athletic Parks,Glenmore Athletic Park,Athletic Park,NaN,5300 19 ST SW,Municipal,Recreation,Water Resources,Recreation,GLENMORE PARK,SOUTH
...,...,...,...,...,...,...,...,...,...,...,...
107,Aquatic & Fitness Centres,Canyon Meadows Aquatic & Fitness Centre,Aquatic & Fitness Centre,NaN,89 CANOVA RD SW,Municipal,Recreation,Recreation,Recreation,CANYON MEADOWS,SOUTH
108,Athletic Parks,Pop Davies Athletic Park,Athletic Park,NaN,6415 OGDEN RD SE,Municipal,Recreation,Recreation,Recreation,OGDEN,SOUTHEAST
109,NaN,Bowview Outdoor Pool & Wading Pool,Outdoor Pool,NaN,1910 6 AV NW,Not-for-Profit,COSPA,Recreation,COSPA,WEST HILLHURST,CENTRE
110,Athletic Parks,Acadia Athletic Park,Athletic Park,NaN,295 90 AV SE,Municipal,Recreation,Recreation,Recreation,ACADIA,SOUTH


In [145]:
community = pd.read_csv("dataset/Historical_Community_Populations_20260305.csv")
community_cols = ['name', 'population', 'occupied dwellings',
       'persons per unit']

for col in community.columns:
    if col not in community_cols:
        community.drop(col, axis=1, inplace=True)

community = community.rename(columns={"name": "COMMUNITY", 
                                      "population":"POPULATION",
                                      "occupied dwellings":"OCCUPIED_DWELLINGS",
                                      "persons per unit":"PERSONS_PER_UNIT"
                                      })
#MERGING existing dataset with new dataset
# Merge on a column
merged_with_pop = pd.merge(recreation_facilities, community, on="COMMUNITY", how="left")  # left join
merged_with_pop['COMMUNITY'] = merged_with_pop['COMMUNITY'].str.title()
# other options for how: 'inner', 'right', 'outer'

# Save to CSV
merged_with_pop.to_csv("recreation_merged_with_pop.csv", index=False)
merged_with_pop

,COMPLEX_CATEGORY,COMPLEX_NAME,FACILITY_TYPE,FACILITY_NAME,ADDRESS,TYP_OF_ORG,ORG,STEWARD,OPER,COMMUNITY,SECTOR,POPULATION,OCCUPIED_DWELLINGS,PERSONS_PER_UNIT
0,NaN,Carrington Skatepark,Skate Park,NaN,37 CARRINGTON PZ NW,Municipal,Parks,Parks,Parks,Carrington,NORTH,NaN,NaN,NaN
1,Arenas (except Leisure Centre arenas),Ernie Starr Arena,Arena,Ice,4808 14 AV SE,Municipal,Recreation,Recreation,Recreation,Forest Lawn,EAST,"7,812","3,078",2.54
2,Arenas (except Leisure Centre arenas),Ernie Starr Arena,Arena,Ice,4808 14 AV SE,Municipal,Recreation,Recreation,Recreation,Forest Lawn,EAST,"8,438","2,974",2.84
3,Arenas (except Leisure Centre arenas),Ernie Starr Arena,Arena,Ice,4808 14 AV SE,Municipal,Recreation,Recreation,Recreation,Forest Lawn,EAST,"8,252","2,868",2.88
4,Arenas (except Leisure Centre arenas),Ernie Starr Arena,Arena,Ice,4808 14 AV SE,Municipal,Recreation,Recreation,Recreation,Forest Lawn,EAST,"8,814","2,828",3.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4713,NaN,Forest Lawn Outdoor Pool & Wading Pool,Outdoor Pool,NaN,1706 39 ST SE,Not-for-Profit,COSPA,Recreation,COSPA,Forest Lawn,EAST,"8,185","2,945",2.78
4714,NaN,Forest Lawn Outdoor Pool & Wading Pool,Outdoor Pool,NaN,1706 39 ST SE,Not-for-Profit,COSPA,Recreation,COSPA,Forest Lawn,EAST,"7,934","2,897",2.74
4715,NaN,Forest Lawn Outdoor Pool & Wading Pool,Outdoor Pool,NaN,1706 39 ST SE,Not-for-Profit,COSPA,Recreation,COSPA,Forest Lawn,EAST,"9,088","2,951",3.08
4716,NaN,Forest Lawn Outdoor Pool & Wading Pool,Outdoor Pool,NaN,1706 39 ST SE,Not-for-Profit,COSPA,Recreation,COSPA,Forest Lawn,EAST,"7,870","2,192",3.59


In [168]:
#cleaning numeric cols
density_pop = pd.read_csv('recreation_merged_with_pop_density.csv')
numeric_cols = ['POPULATION', 'OCCUPIED_DWELLINGS', 'PERSONS_PER_UNIT','Area (km²)', 'Population Density (people/km²)']

for col in numeric_cols:
    density_pop[col] = (
        density_pop[col]
        .astype(str)             # in case some are already numeric
        .str.replace(',', '')    # remove commas
        .replace('nan', '')      # optional: remove string 'nan'
    )
    density_pop[col] = pd.to_numeric(density_pop[col], errors='coerce')

density_pop.isna().sum()

COMPLEX_CATEGORY                   2459
COMPLEX_NAME                          0
FACILITY_TYPE                         0
FACILITY_NAME                      3662
ADDRESS                               0
TYP_OF_ORG                            0
ORG                                   1
STEWARD                               0
OPER                                  0
COMMUNITY                             0
SECTOR                                0
POPULATION                           18
OCCUPIED_DWELLINGS                   18
PERSONS_PER_UNIT                     18
Area (km²)                          912
Population Density (people/km²)     912
dtype: int64

In [ ]:
cols = [
    'University of Calgary', 'Douglasdale/Glen', 'Albert Park/Raddisson Height',
    'Killarney/Glengary', 'Midnapore', 'Ogden', 'Downtown', 'Parkhill', 'Shawnessy'
    ]

areas = {
    'University of Calgary': 0.8,
    'Douglasdale/Glen':5.1,
    'Albert Park/Raddisson Height':2.5,
    'Killarney/Glengary':1.8,
    'Midnapore':2.9,
    'Ogden':1.6,
    'Downtown':1.9, 
    'Parkhill':0.6,
    'Shawnessy':3.7
}

population = {
    'University of Calgary': 33000,
    'Douglasdale/Glen':11.768,
    'Albert Park/Raddisson Height':6234,
    'Killarney/Glengary':6543,
    'Midnapore':6888,
    'Ogden':8315,
    'Downtown':13000, 
    'Parkhill':1543,
    'Shawnessy':9055
}

density_pop['Area (km²)'] = density_pop['Area (km²)'].fillna(
    density_pop['COMMUNITY'].map(areas)
)

density_pop['POPULATION'] = density_pop['POPULATION'].fillna(
    density_pop['COMMUNITY'].map(population)
)

In [179]:
density_pop['Area (km²)'] = density_pop['Area (km²)'].fillna(4.0)
density_pop['PERSONS_PER_UNIT'] = density_pop['PERSONS_PER_UNIT'].fillna(2.0)
density_pop['OCCUPIED_DWELLINGS'] = density_pop['OCCUPIED_DWELLINGS'].fillna(2803)
density_pop['POPULATION'] = density_pop['POPULATION'].fillna(3112)
density_pop['ORG'] = density_pop['ORG'].fillna('Parks')

columns_to_drop = ['COMPLEX_CATEGORY', 'FACILITY_NAME']

for col in density_pop:
    if col in columns_to_drop:
        density_pop.drop(col, axis=1, inplace=True)

final_data = density_pop.to_csv("final_data/recreation_data.csv", index=False)